# Pattern Recognition: Image De-noising System
## Comparative Empirical Evaluation of Classical Filtering, Subspace Learning (PCA), and Deep Convolutional Autoencoders

---
* **Project Authors:** Pratiyush Sharma, Akshada Shinde, Ayushi Kadam, Pushpak Agarwal
* **Department:** Department of Computer Science and Engineering
* **Problem Domain:** Image De-noising in Medical Imaging, Low-Light Photography, and Surveillance
* **Dataset:** Fashion-MNIST (Grayscale $28 \times 28$ high-texture garment patterns)
* **Corruption Models:**
  1. Additive White Gaussian Noise (AWGN): $\sigma \in \{15, 25, 50\}$
  2. Impulse Salt-and-Pepper (S&P) Noise: $d \in \{0.05, 0.10\}$
* **Evaluated Paradigms:**
  1. *Classical Spatial Filtering:* Gaussian Blur, 2D Median Filter, Edge-Preserving Bilateral Filter
  2. *Statistical Subspace Reconstruction:* Principal Component Analysis (PCA) with validation-based component selection
  3. *Deep Feature Learning:* Convolutional Denoising Autoencoder (CDAE) in PyTorch
* **Evaluation Metrics:** Peak Signal-to-Noise Ratio (PSNR), Structural Similarity Index (SSIM), Mean Squared Error (MSE), Inference Latency (ms/image)
* **Reproducibility:** Random seed fixed to `42` across all stochastic generators.


In [1]:
# Environment Setup, Directory Creation, and Library Imports
import os
import sys
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cv2

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# Fix random seed for deterministic reproducibility
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

from sklearn.decomposition import PCA
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

# Create directories
FIGURES_DIR = os.path.join(os.getcwd(), 'figures')
RESULTS_DIR = os.path.join(os.getcwd(), 'results')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Plot formatting setup
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment: PyTorch {torch.__version__} | OpenCV {cv2.__version__} | Device: {device}")


Environment: PyTorch 2.11.0+cpu | OpenCV 5.0.0 | Device: cpu


## 1. Dataset Loading and Partitioning (Fashion-MNIST)

We employ the Fashion-MNIST dataset, which contains complex high-frequency contours, fabric textures, stitching, and distinct silhouette variations.

We partition the dataset into:
- **Training Set (first 8,000 images):** Used to fit the statistical PCA basis and train the CDAE.
- **Validation Set (images 8,000-9,000 of training split):** Used for filter tuning, PCA component selection ($k$), and CDAE validation monitoring.
- **Test Set (first 500 images of test split):** Reserved strictly for unbiased quantitative benchmarking across all noise conditions.


In [2]:
# Download and partition Fashion-MNIST
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Extract images as float32 arrays normalized in [0, 1]
X_train_clean = train_dataset.data[:8000].numpy().astype(np.float32) / 255.0
X_val_clean = train_dataset.data[8000:9000].numpy().astype(np.float32) / 255.0
X_test_clean = test_dataset.data[:500].numpy().astype(np.float32) / 255.0

print(f"Partitions: Train={X_train_clean.shape}, Val={X_val_clean.shape}, Test={X_test_clean.shape}")
print(f"Pixel Intensity Range: [{X_train_clean.min():.1f}, {X_train_clean.max():.1f}]")


Partitions: Train=(8000, 28, 28), Val=(1000, 28, 28), Test=(500, 28, 28)
Pixel Intensity Range: [0.0, 1.0]


## 2. Noise Corruption Generator Functions

### Mathematical Formulations

#### 1. Additive White Gaussian Noise (AWGN)
Gaussian noise models thermal agitation of charge carriers in image sensors:
$$I_{\text{noisy}}(x, y) = \text{clip}\Big(I(x, y) + \eta(x, y), 0, 1\Big), \quad \eta \sim \mathcal{N}(0, \sigma^2)$$
where $\sigma \in \{15, 25, 50\}$ on the standard $[0, 255]$ scale (normalized to $\sigma_{\text{norm}} = \sigma / 255$).

#### 2. Salt-and-Pepper (Impulse) Noise
Impulse noise models bit errors during transmission:
$$I_{\text{noisy}}(x, y) = \begin{cases} 
1.0, & \text{with probability } d / 2 \quad (\text{Salt}) \\ 
0.0, & \text{with probability } d / 2 \quad (\text{Pepper}) \\ 
I(x, y), & \text{with probability } 1 - d \quad (\text{Clean})
\end{cases}$$
where $d \in \{0.05, 0.10\}$ denotes corruption density.


In [3]:
# Define noise synthesis functions
def add_gaussian_noise(images, sigma_255):
    sigma = sigma_255 / 255.0
    noise = np.random.normal(loc=0.0, scale=sigma, size=images.shape).astype(np.float32)
    return np.clip(images + noise, 0.0, 1.0)

def add_salt_and_pepper_noise(images, density):
    noisy = images.copy()
    num_salt = int(np.ceil(density * images.size * 0.5))
    num_pepper = int(np.ceil(density * images.size * 0.5))

    salt_coords = [np.random.randint(0, i, num_salt) for i in images.shape]
    noisy[tuple(salt_coords)] = 1.0

    pepper_coords = [np.random.randint(0, i, num_pepper) for i in images.shape]
    noisy[tuple(pepper_coords)] = 0.0
    return noisy

# Define evaluation conditions
NOISE_CONDITIONS = [
    {"name": "Gaussian (sigma=15)", "display": r"Gaussian ($\sigma=15$)", "type": "gaussian", "param": 15},
    {"name": "Gaussian (sigma=25)", "display": r"Gaussian ($\sigma=25$)", "type": "gaussian", "param": 25},
    {"name": "Gaussian (sigma=50)", "display": r"Gaussian ($\sigma=50$)", "type": "gaussian", "param": 50},
    {"name": "S&P (d=0.05)",        "display": "S&P (d=0.05)",            "type": "sp",       "param": 0.05},
    {"name": "S&P (d=0.10)",        "display": "S&P (d=0.10)",            "type": "sp",       "param": 0.10},
]

# Generate test noisy sets FIRST right after seeding to guarantee identical test sets
test_noisy_dict = {}
for cond in NOISE_CONDITIONS:
    if cond["type"] == "gaussian":
        test_noisy_dict[cond["name"]] = add_gaussian_noise(X_test_clean, cond["param"])
    else:
        test_noisy_dict[cond["name"]] = add_salt_and_pepper_noise(X_test_clean, cond["param"])

# Generate validation noisy sets (200 images) for hyperparameter tuning
val_noisy_dict = {}
val_clean_200 = X_val_clean[:200]
for cond in NOISE_CONDITIONS:
    if cond["type"] == "gaussian":
        val_noisy_dict[cond["name"]] = add_gaussian_noise(val_clean_200, cond["param"])
    else:
        val_noisy_dict[cond["name"]] = add_salt_and_pepper_noise(val_clean_200, cond["param"])

print(f"Generated {len(test_noisy_dict)} test sets and {len(val_noisy_dict)} validation sets.")


Generated 5 test sets and 5 validation sets.


## 3. Evaluation Metrics Formulation

1. **Mean Squared Error (MSE):**
   $$\text{MSE} = \frac{1}{M \times N} \sum_{i=1}^{M} \sum_{j=1}^{N} \Big(I(i, j) - \hat{I}(i, j)\Big)^2$$

2. **Peak Signal-to-Noise Ratio (PSNR):**
   $$\text{PSNR} = 10 \log_{10} \left( \frac{\text{MAX}_I^2}{\text{MSE}} \right) = 10 \log_{10}\left(\frac{1.0}{\text{MSE}}\right) \quad \text{[dB]}$$
   Reported as the mean of per-image PSNR values across the test batch.

3. **Structural Similarity Index (SSIM):**
   $$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$


In [4]:
# Helper function to compute mean per-image PSNR, SSIM, and MSE
def compute_metrics(clean_imgs, denoised_imgs):
    psnrs, ssims, mses = [], [], []
    for c, d in zip(clean_imgs, denoised_imgs):
        mse_val = float(np.mean((c - d) ** 2))
        psnr_val = float(compute_psnr(c, d, data_range=1.0))
        ssim_val = float(compute_ssim(c, d, data_range=1.0))
        mses.append(mse_val)
        psnrs.append(psnr_val)
        ssims.append(ssim_val)
    return float(np.mean(psnrs)), float(np.mean(ssims)), float(np.mean(mses))


## 4. Solution 1: Classical Filtering & Hyperparameter Tuning

We implement Gaussian, Median, and Bilateral filters. Hyperparameters are tuned on the validation split ($200$ images) separately for each noise condition by maximizing validation PSNR:
- **Gaussian:** $\text{ksize} \in \{3, 5\}$, $\sigma \in \{0.5, 0.8, 1.0, 1.5\}$
- **Median:** $\text{ksize} \in \{3, 5\}$
- **Bilateral:** $d \in \{5, 7, 9\}$, $\sigma_{\text{color}} \in \{0.05, 0.1, 0.15, 0.25\}$, $\sigma_{\text{space}} \in \{1.5, 3.0\}$


In [5]:
# Tune filters on validation split (200 images) per condition
filter_tuning_results = {}

for cond in NOISE_CONDITIONS:
    cname = cond["name"]
    v_noisy = val_noisy_dict[cname]
    filter_tuning_results[cname] = {}
    
    # 1. Gaussian Tuning
    best_g_p = -1.0
    best_g = {}
    for ks in [3, 5]:
        for sig in [0.5, 0.8, 1.0, 1.5]:
            den = np.array([cv2.GaussianBlur(img, (ks, ks), sigmaX=sig, sigmaY=sig) for img in v_noisy])
            p = float(np.mean([compute_psnr(c, d, data_range=1.0) for c, d in zip(val_clean_200, np.clip(den, 0, 1))]))
            if p > best_g_p:
                best_g_p = p
                best_g = {"ksize": ks, "sigma": sig, "val_psnr": round(p, 2)}
    filter_tuning_results[cname]["Gaussian Filter"] = best_g
    
    # 2. Median Tuning
    best_m_p = -1.0
    best_m = {}
    for ks in [3, 5]:
        den = np.array([cv2.medianBlur(img, ks) for img in v_noisy])
        p = float(np.mean([compute_psnr(c, d, data_range=1.0) for c, d in zip(val_clean_200, np.clip(den, 0, 1))]))
        if p > best_m_p:
            best_m_p = p
            best_m = {"ksize": ks, "val_psnr": round(p, 2)}
    filter_tuning_results[cname]["Median Filter"] = best_m
    
    # 3. Bilateral Tuning
    best_b_p = -1.0
    best_b = {}
    for d in [5, 7, 9]:
        for sc in [0.05, 0.1, 0.15, 0.25]:
            for ss in [1.5, 3.0]:
                den = np.array([cv2.bilateralFilter(img, d=d, sigmaColor=sc, sigmaSpace=ss) for img in v_noisy])
                p = float(np.mean([compute_psnr(c, d, data_range=1.0) for c, d in zip(val_clean_200, np.clip(den, 0, 1))]))
                if p > best_b_p:
                    best_b_p = p
                    best_b = {"d": d, "sigma_color": sc, "sigma_space": ss, "val_psnr": round(p, 2)}
    filter_tuning_results[cname]["Bilateral Filter"] = best_b

with open(os.path.join(RESULTS_DIR, 'filter_tuning.json'), 'w') as f:
    json.dump(filter_tuning_results, f, indent=2)

print("Tuned Hyperparameters Saved:")
for cname, res in filter_tuning_results.items():
    print(f"  {cname}: G={res['Gaussian Filter']}, M={res['Median Filter']}, B={res['Bilateral Filter']}")

def apply_tuned_gaussian(images, cname):
    p = filter_tuning_results[cname]["Gaussian Filter"]
    return np.clip(np.array([cv2.GaussianBlur(img, (p["ksize"], p["ksize"]), sigmaX=p["sigma"], sigmaY=p["sigma"]) for img in images]), 0, 1)

def apply_tuned_median(images, cname):
    p = filter_tuning_results[cname]["Median Filter"]
    return np.clip(np.array([cv2.medianBlur(img, p["ksize"]) for img in images]), 0, 1)

def apply_tuned_bilateral(images, cname):
    p = filter_tuning_results[cname]["Bilateral Filter"]
    return np.clip(np.array([cv2.bilateralFilter(img, d=p["d"], sigmaColor=p["sigma_color"], sigmaSpace=p["sigma_space"]) for img in images]), 0, 1)


Tuned Hyperparameters Saved:
  Gaussian (sigma=15): G={'ksize': 3, 'sigma': 0.5, 'val_psnr': 24.88}, M={'ksize': 3, 'val_psnr': 21.04}, B={'d': 7, 'sigma_color': 0.1, 'sigma_space': 1.5, 'val_psnr': 28.33}
  Gaussian (sigma=25): G={'ksize': 3, 'sigma': 0.5, 'val_psnr': 22.58}, M={'ksize': 3, 'val_psnr': 20.27}, B={'d': 5, 'sigma_color': 0.15, 'sigma_space': 3.0, 'val_psnr': 24.51}
  Gaussian (sigma=50): G={'ksize': 5, 'sigma': 0.5, 'val_psnr': 18.23}, M={'ksize': 3, 'val_psnr': 18.16}, B={'d': 7, 'sigma_color': 0.25, 'sigma_space': 3.0, 'val_psnr': 19.29}
  S&P (d=0.05): G={'ksize': 5, 'sigma': 0.5, 'val_psnr': 19.73}, M={'ksize': 3, 'val_psnr': 20.92}, B={'d': 9, 'sigma_color': 0.25, 'sigma_space': 3.0, 'val_psnr': 17.44}
  S&P (d=0.10): G={'ksize': 3, 'sigma': 0.8, 'val_psnr': 17.84}, M={'ksize': 3, 'val_psnr': 20.14}, B={'d': 9, 'sigma_color': 0.25, 'sigma_space': 3.0, 'val_psnr': 14.66}


## 5. Solution 2: Statistical Subspace Reconstruction (PCA)

We project $784$-dimensional image vectors onto the top-$k$ principal components derived from the clean training set covariance matrix.
The parameter $k$ is selected by evaluating reconstruction PSNR on the validation set under Gaussian noise ($\sigma=25$) across the extended candidate set:
$$k \in [10, 20, 35, 50, 75, k_{90}, 120, k_{95}, 220, 300, 400, 500, 600]$$


In [6]:
# PCA Subspace Fitting and Extended Component Optimization
X_train_flat = X_train_clean.reshape(len(X_train_clean), -1)

# Fit initial PCA up to 600 components to cover k_90, k_95, and beyond
pca_estimator = PCA(n_components=600, random_state=RANDOM_SEED)
pca_estimator.fit(X_train_flat)

cum_var = np.cumsum(pca_estimator.explained_variance_ratio_)
k_90 = int(np.where(cum_var >= 0.90)[0][0] + 1)
k_95 = int(np.where(cum_var >= 0.95)[0][0] + 1)
print(f"Variance Thresholds: 90% at k={k_90}, 95% at k={k_95}")

k_candidates = sorted(list(set([10, 20, 35, 50, 75, k_90, 120, k_95, 220, 300, 400, 500, 600])))
k_psnr_sweep = []
sample_noisy_val = val_noisy_dict["Gaussian (sigma=25)"].reshape(200, -1)
sample_clean_val = val_clean_200

# Optimal k is selected by maximizing reconstruction PSNR on the validation set under Gaussian sigma=25 (rather than by an arbitrary variance threshold).
for k in k_candidates:
    pca_k = PCA(n_components=k, random_state=RANDOM_SEED)
    pca_k.fit(X_train_flat)
    val_recon = pca_k.inverse_transform(pca_k.transform(sample_noisy_val)).reshape(-1, 28, 28)
    val_recon = np.clip(val_recon, 0.0, 1.0)
    p = float(np.mean([compute_psnr(c, d, data_range=1.0) for c, d in zip(sample_clean_val, val_recon)]))
    k_psnr_sweep.append(p)
    print(f"  k={k:3d} (Variance: {cum_var[k-1]*100:.1f}%) -> Val PSNR: {p:.2f} dB")

best_k = k_candidates[int(np.argmax(k_psnr_sweep))]
pca_mname = f"PCA (k={best_k})"
print(f"Selected Optimal PCA Dimension: {pca_mname} (Val PSNR = {max(k_psnr_sweep):.2f} dB)")

# Fit final PCA model
pca_model = PCA(n_components=best_k, random_state=RANDOM_SEED)
t_pca_start = time.perf_counter()
pca_model.fit(X_train_flat)
pca_fit_time = time.perf_counter() - t_pca_start
print(f"Fitted final PCA model in {pca_fit_time:.2f} seconds.")

def denoise_pca(images, model):
    n = len(images)
    flat = images.reshape(n, -1)
    recon = model.inverse_transform(model.transform(flat)).reshape(n, 28, 28)
    return np.clip(recon, 0.0, 1.0)


Variance Thresholds: 90% at k=82, 95% at k=181


  k= 10 (Variance: 72.0%) -> Val PSNR: 16.57 dB


  k= 20 (Variance: 78.6%) -> Val PSNR: 17.66 dB


  k= 35 (Variance: 83.5%) -> Val PSNR: 18.73 dB


  k= 50 (Variance: 86.4%) -> Val PSNR: 19.45 dB


  k= 75 (Variance: 89.4%) -> Val PSNR: 20.31 dB


... (4 outputs left out)

  k=300 (Variance: 97.6%) -> Val PSNR: 22.72 dB
  k=400 (Variance: 98.7%) -> Val PSNR: 22.81 dB


  k=500 (Variance: 99.4%) -> Val PSNR: 22.67 dB


  k=600 (Variance: 99.8%) -> Val PSNR: 22.38 dB
Selected Optimal PCA Dimension: PCA (k=400) (Val PSNR = 22.81 dB)


Fitted final PCA model in 0.23 seconds.


## 6. Solution 3: Convolutional Denoising Autoencoder (CDAE)

The CDAE utilizes 2D convolutional layers, batch normalization, and transpose convolutions ($66,593$ parameters).


In [7]:
# PyTorch CDAE Architecture & Checkpoint Loading
class ConvDenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(ConvDenoisingAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

cdae_model = ConvDenoisingAutoencoder().to(device)

cdae_weights_path = os.path.join(RESULTS_DIR, 'cdae_weights.pth')
history_path = os.path.join(RESULTS_DIR, 'cdae_history.json')

if os.path.exists(cdae_weights_path) and os.path.exists(history_path):
    print("Loading pre-trained CDAE checkpoint weights (no retraining)...")
    cdae_model.load_state_dict(torch.load(cdae_weights_path, map_location=device))
    with open(history_path, 'r') as hf:
        hist_data = json.load(hf)
        train_loss_history = hist_data['train_loss']
        val_loss_history = hist_data['val_loss']
        cdae_total_train_time = hist_data['train_time_s']
    print(f"Loaded CDAE checkpoint (Training duration: {cdae_total_train_time:.1f}s).")
else:
    raise FileNotFoundError("cdae_weights.pth not found in results directory!")

def denoise_cdae(images, model):
    model.eval()
    n = len(images)
    tensor_in = torch.tensor(images[:, np.newaxis, :, :], dtype=torch.float32).to(device)
    out_list = []
    batch_size = 128
    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch = tensor_in[i:i+batch_size]
            recon = model(batch)
            out_list.append(recon.cpu().numpy())
    result = np.concatenate(out_list, axis=0).squeeze(1)
    return np.clip(result, 0.0, 1.0)


Loading pre-trained CDAE checkpoint weights (no retraining)...
Loaded CDAE checkpoint (Training duration: 123.4s).


## 7. Comprehensive Quantitative Benchmarking

We evaluate all methods across the 500 test images for all five noise configurations.


In [8]:
# Comprehensive Benchmarking Execution
benchmark_records = []
denoised_cache = {}

for cond in NOISE_CONDITIONS:
    cname = cond["name"]
    X_noisy = test_noisy_dict[cname]
    denoised_cache[cname] = {}
    
    cond_methods = [
        {"name": "Noisy Input",      "func": lambda x: x},
        {"name": "Gaussian Filter",  "func": lambda x, cn=cname: apply_tuned_gaussian(x, cn)},
        {"name": "Median Filter",    "func": lambda x, cn=cname: apply_tuned_median(x, cn)},
        {"name": "Bilateral Filter", "func": lambda x, cn=cname: apply_tuned_bilateral(x, cn)},
        {"name": pca_mname,          "func": lambda x: denoise_pca(x, pca_model)},
        {"name": "CDAE",             "func": lambda x: denoise_cdae(x, cdae_model)},
    ]
    
    for m in cond_methods:
        mname = m["name"]
        t_start = time.perf_counter()
        denoised = m["func"](X_noisy)
        elapsed = time.perf_counter() - t_start
        per_img_ms = (elapsed / len(X_noisy)) * 1000.0

        p_val, s_val, mse_val = compute_metrics(X_test_clean, denoised)
        denoised_cache[cname][mname] = denoised
        benchmark_records.append({
            "Noise Condition": cname,
            "Method": mname,
            "PSNR (dB)": round(p_val, 2),
            "SSIM": round(s_val, 4),
            "MSE": round(mse_val, 5),
            "Runtime (ms/img)": round(per_img_ms, 3)
        })

df_benchmark = pd.DataFrame(benchmark_records)
df_benchmark.to_csv(os.path.join(RESULTS_DIR, 'metrics_summary.csv'), index=False)

# Compute per-condition best method by PSNR and SSIM
best_by_psnr = {}
best_by_ssim = {}
for cond in NOISE_CONDITIONS:
    cname = cond["name"]
    cond_df = df_benchmark[df_benchmark["Noise Condition"] == cname]
    denoiser_df = cond_df[cond_df["Method"] != "Noisy Input"]
    
    top_p = denoiser_df.loc[denoiser_df["PSNR (dB)"].idxmax()]
    best_by_psnr[cname] = {"method": str(top_p["Method"]), "psnr": float(top_p["PSNR (dB)"])}
    
    top_s = denoiser_df.loc[denoiser_df["SSIM"].idxmax()]
    best_by_ssim[cname] = {"method": str(top_s["Method"]), "ssim": float(top_s["SSIM"])}

summary_json = {
    "records": benchmark_records,
    "pca_fit_time_s": round(pca_fit_time, 2),
    "cdae_train_time_s": round(cdae_total_train_time, 2),
    "pca_best_k": best_k,
    "pca_mname": pca_mname,
    "train_loss_history": [round(float(x), 6) for x in train_loss_history],
    "val_loss_history": [round(float(x), 6) for x in val_loss_history],
    "k_candidates": [int(x) for x in k_candidates],
    "k_psnr_sweep": [round(float(x), 2) for x in k_psnr_sweep],
    "cum_var_ratio": [round(float(cum_var[k-1]), 4) for k in k_candidates],
    "best_by_psnr": best_by_psnr,
    "best_by_ssim": best_by_ssim
}
with open(os.path.join(RESULTS_DIR, 'metrics_summary.json'), 'w') as f:
    json.dump(summary_json, f, indent=2)

# Display formatted table
pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 1000)
print("\nFinal Quantitative Benchmark Results:")
display(df_benchmark)



Final Quantitative Benchmark Results:


,Noise Condition,Method,PSNR (dB),SSIM,MSE,Runtime (ms/img)
0,Gaussian (sigma=15),Noisy Input,26.09,0.8049,0.00248,0.000
1,Gaussian (sigma=15),Gaussian Filter,24.86,0.8081,0.00337,0.015
2,Gaussian (sigma=15),Median Filter,21.04,0.7335,0.00898,0.005
3,Gaussian (sigma=15),Bilateral Filter,28.35,0.8481,0.00149,0.046
4,Gaussian (sigma=15),PCA (k=400),25.97,0.8054,0.00269,0.017
5,Gaussian (sigma=15),CDAE,24.61,0.8566,0.00404,0.367
6,Gaussian (sigma=25),Noisy Input,21.78,0.7273,0.00669,0.000
7,Gaussian (sigma=25),Gaussian Filter,22.56,0.7480,0.00563,0.008
8,Gaussian (sigma=25),Median Filter,20.26,0.6787,0.01038,0.004
9,Gaussian (sigma=25),Bilateral Filter,24.52,0.7835,0.00359,0.015


## 8. High-Resolution Visualizations & Figures Generation

The code below saves all 9 publication-grade figures at 300 DPI to `./figures/`.


In [9]:
# Generate and Save All 9 Figures at 300 DPI
all_eval_methods = ["Noisy Input", "Gaussian Filter", "Median Filter", "Bilateral Filter", pca_mname, "CDAE"]
method_colors = {
    "Noisy Input": "#9E9E9E",
    "Gaussian Filter": "#4A90E2",
    "Median Filter": "#50E3C2",
    "Bilateral Filter": "#F5A623",
    pca_mname: "#9013FE",
    "CDAE": "#D0021B"
}

# 1. Sample Clean, Noisy, and De-noised Images
fig, axes = plt.subplots(4, 7, figsize=(15, 9), dpi=300)
test_sample_indices = [0, 1, 3, 4]
eval_conditions_show = ["Gaussian (sigma=25)", "S&P (d=0.05)", "Gaussian (sigma=50)", "S&P (d=0.10)"]
eval_labels_show = [r"Gaussian ($\sigma=25$)", "S&P (d=0.05)", r"Gaussian ($\sigma=50$)", "S&P (d=0.10)"]

for row_idx, (img_idx, cond_name, cond_lbl) in enumerate(zip(test_sample_indices, eval_conditions_show, eval_labels_show)):
    axes[row_idx, 0].imshow(X_test_clean[img_idx], cmap='gray', vmin=0, vmax=1)
    if row_idx == 0:
        axes[row_idx, 0].set_title("Ground Truth (Clean)", fontsize=10, fontweight='bold', pad=8)
    axes[row_idx, 0].set_ylabel(f"{cond_lbl}", fontsize=10, fontweight='bold')
    axes[row_idx, 0].set_xticks([])
    axes[row_idx, 0].set_yticks([])

    for col_idx, mname in enumerate(all_eval_methods):
        ax = axes[row_idx, col_idx + 1]
        img_show = denoised_cache[cond_name][mname][img_idx]
        ax.imshow(img_show, cmap='gray', vmin=0, vmax=1)
        if row_idx == 0:
            ax.set_title(mname, fontsize=10, fontweight='bold', pad=8)
        p_val = compute_psnr(X_test_clean[img_idx], img_show, data_range=1.0)
        ax.set_xlabel(f"{p_val:.1f} dB", fontsize=8, labelpad=2)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'sample_denoised_comparisons.png'), bbox_inches='tight')
plt.close(fig)

# 2. Zoomed Detail Crops (Includes Bilateral Filter)
fig, axes = plt.subplots(2, 7, figsize=(16, 5.5), dpi=300)
crop_img_idx = 0
crop_cond = "Gaussian (sigma=25)"
crop_y1, crop_y2, crop_x1, crop_x2 = 12, 26, 8, 22

full_items = [
    ("Clean Reference", X_test_clean[crop_img_idx]),
    (r"Noisy ($\sigma=25$)", test_noisy_dict[crop_cond][crop_img_idx]),
    ("Gaussian Filter", denoised_cache[crop_cond]["Gaussian Filter"][crop_img_idx]),
    ("Median Filter", denoised_cache[crop_cond]["Median Filter"][crop_img_idx]),
    ("Bilateral Filter", denoised_cache[crop_cond]["Bilateral Filter"][crop_img_idx]),
    (pca_mname, denoised_cache[crop_cond][pca_mname][crop_img_idx]),
    ("CDAE", denoised_cache[crop_cond]["CDAE"][crop_img_idx]),
]

for col, (title, img) in enumerate(full_items):
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    rect = Rectangle((crop_x1, crop_y1), crop_x2 - crop_x1, crop_y2 - crop_y1,
                     linewidth=1.5, edgecolor='red', facecolor='none')
    axes[0, col].add_patch(rect)
    axes[0, col].set_title(title, fontsize=10, fontweight='bold')
    axes[0, col].axis('off')

for col, (title, img) in enumerate(full_items):
    crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
    axes[1, col].imshow(crop, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    axes[1, col].set_title("Zoomed ROI", fontsize=9, color='#b30000', fontweight='bold')
    axes[1, col].axis('off')

plt.suptitle("Region of Interest (ROI) Edge Preservation & Artifact Analysis", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'zoomed_detail_crops.png'), bbox_inches='tight')
plt.close(fig)

# 3. PSNR Bar Chart (Includes Noisy Input Baseline)
cond_keys = [c["name"] for c in NOISE_CONDITIONS]
cond_displays = [c["display"] for c in NOISE_CONDITIONS]
x = np.arange(len(cond_keys))
n_m = len(all_eval_methods)
width = 0.13
max_psnr = df_benchmark["PSNR (dB)"].max()

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)
for i, mname in enumerate(all_eval_methods):
    vals = [df_benchmark[(df_benchmark["Noise Condition"] == c) & (df_benchmark["Method"] == mname)]["PSNR (dB)"].values[0] for c in cond_keys]
    offset = (i - (n_m - 1) / 2) * width
    if mname == "Noisy Input":
        ax.bar(x + offset, vals, width, label=mname, color=method_colors[mname], hatch='//', edgecolor='black', linewidth=0.6, alpha=0.8)
    else:
        ax.bar(x + offset, vals, width, label=mname, color=method_colors[mname], edgecolor='black', linewidth=0.6, alpha=0.9)

ax.set_ylabel("PSNR (dB) - Higher is Better", fontsize=11, fontweight='bold')
ax.set_title("Quantitative PSNR Comparison Across Noise Conditions", fontsize=12, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(cond_displays, fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(10, np.ceil(max_psnr) + 2)
ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#cccccc', fontsize=9, loc='upper right')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'psnr_bar_chart.png'), bbox_inches='tight')
plt.close(fig)

# 4. SSIM Bar Chart (Includes Noisy Input Baseline)
fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)
for i, mname in enumerate(all_eval_methods):
    vals = [df_benchmark[(df_benchmark["Noise Condition"] == c) & (df_benchmark["Method"] == mname)]["SSIM"].values[0] for c in cond_keys]
    offset = (i - (n_m - 1) / 2) * width
    if mname == "Noisy Input":
        ax.bar(x + offset, vals, width, label=mname, color=method_colors[mname], hatch='//', edgecolor='black', linewidth=0.6, alpha=0.8)
    else:
        ax.bar(x + offset, vals, width, label=mname, color=method_colors[mname], edgecolor='black', linewidth=0.6, alpha=0.9)

ax.set_ylabel("SSIM (Structural Similarity Index) - Higher is Better", fontsize=11, fontweight='bold')
ax.set_title("Quantitative SSIM Structural Fidelity Comparison", fontsize=12, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(cond_displays, fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, 1.05)
ax.legend(frameon=True, facecolor='#ffffff', edgecolor='#cccccc', fontsize=9, loc='upper right')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'ssim_bar_chart.png'), bbox_inches='tight')
plt.close(fig)

# 5. PSNR vs Noise Level Line Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=300)
g_sigmas = [15, 25, 50]
g_conds = ["Gaussian (sigma=15)", "Gaussian (sigma=25)", "Gaussian (sigma=50)"]
markers = {'Noisy Input': 'x', 'Gaussian Filter': 'o', 'Median Filter': 's', 'Bilateral Filter': '^', pca_mname: 'd', 'CDAE': '*'}

for mname in all_eval_methods:
    y_vals = [df_benchmark[(df_benchmark["Noise Condition"] == c) & (df_benchmark["Method"] == mname)]["PSNR (dB)"].values[0] for c in g_conds]
    linestyle = '--' if mname == 'Noisy Input' else '-'
    ax1.plot(g_sigmas, y_vals, marker=markers[mname], linestyle=linestyle, linewidth=2, markersize=7, label=mname, color=method_colors[mname])

ax1.set_xlabel(r"Gaussian Standard Deviation ($\sigma$)", fontsize=10, fontweight='bold')
ax1.set_ylabel("PSNR (dB)", fontsize=10, fontweight='bold')
ax1.set_title("(a) De-noising Performance: Gaussian AWGN", fontsize=11, fontweight='bold')
ax1.set_xticks(g_sigmas)
ax1.set_ylim(bottom=10)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=8, loc='upper right')

sp_densities = [0.05, 0.10]
sp_conds = ["S&P (d=0.05)", "S&P (d=0.10)"]

for mname in all_eval_methods:
    y_vals = [df_benchmark[(df_benchmark["Noise Condition"] == c) & (df_benchmark["Method"] == mname)]["PSNR (dB)"].values[0] for c in sp_conds]
    linestyle = '--' if mname == 'Noisy Input' else '-'
    ax2.plot([d * 100 for d in sp_densities], y_vals, marker=markers[mname], linestyle=linestyle, linewidth=2, markersize=7, label=mname, color=method_colors[mname])

ax2.set_xlabel("Impulse Noise Density (%)", fontsize=10, fontweight='bold')
ax2.set_ylabel("PSNR (dB)", fontsize=10, fontweight='bold')
ax2.set_title("(b) De-noising Performance: Salt & Pepper Noise", fontsize=11, fontweight='bold')
ax2.set_xticks([5, 10])
ax2.set_ylim(bottom=10)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=8, loc='upper right')

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'psnr_vs_noise_level.png'), bbox_inches='tight')
plt.close(fig)

# 6. PCA Cumulative Variance & PSNR Curve
fig, ax1 = plt.subplots(figsize=(9, 5), dpi=300)
color1 = '#1f77b4'
color2 = '#d62728'
cum_var_sweep = [float(cum_var[k-1]) * 100 for k in k_candidates]
ax1.plot(k_candidates, cum_var_sweep, marker='o', color=color1, linewidth=2.2, label='Cumulative Variance (%)')
ax1.axhline(90, color=color1, linestyle=':', alpha=0.7, label=f'90% Variance (k={k_90})')
ax1.axhline(95, color=color1, linestyle='--', alpha=0.7, label=f'95% Variance (k={k_95})')
ax1.axvline(best_k, color='gray', linestyle='-.', alpha=0.9, label=f'Optimal k={best_k} (by Val PSNR)')
ax1.set_xlabel("Number of Principal Components (k)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Cumulative Explained Variance (%)", color=color1, fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(k_candidates, k_psnr_sweep, marker='s', color=color2, linewidth=2.2, linestyle='-', label=r'Validation PSNR (dB) on $\sigma=25$')
ax2.set_ylabel(r"Validation PSNR (dB) on Gaussian ($\sigma=25$)", color=color2, fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', frameon=True, fontsize=8.5)
plt.title("PCA Dimensionality Selection: Cumulative Variance vs Reconstruction PSNR", fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'pca_variance_and_psnr.png'), bbox_inches='tight')
plt.close(fig)

# 7. CDAE Loss Curves
fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=300)
epochs_arr = range(1, len(train_loss_history) + 1)
ax.plot(epochs_arr, train_loss_history, marker='o', linewidth=2.2, color='#2b5c8f', label='Training MSE Loss')
ax.plot(epochs_arr, val_loss_history, marker='s', linewidth=2.2, color='#e05a47', label='Validation MSE Loss')
ax.set_xlabel("Epoch Number", fontsize=11, fontweight='bold')
ax.set_ylabel("Mean Squared Error (MSE)", fontsize=11, fontweight='bold')
ax.set_title("CDAE Convergence Trajectory Over 10 Epochs", fontsize=12, fontweight='bold', pad=10)
ax.set_xticks(list(epochs_arr))
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(frameon=True, fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'cdae_loss_curves.png'), bbox_inches='tight')
plt.close(fig)

# 8. Runtime Comparison
fig, ax = plt.subplots(figsize=(9, 4.8), dpi=300)
runtime_methods = ["Gaussian Filter", "Median Filter", "Bilateral Filter", pca_mname, "CDAE"]
avg_latencies = [df_benchmark[df_benchmark["Method"] == m]["Runtime (ms/img)"].mean() for m in runtime_methods]
colors_run = [method_colors[m] for m in runtime_methods]

bars = ax.barh(runtime_methods, avg_latencies, color=colors_run, edgecolor='black', linewidth=0.6, alpha=0.85)
ax.set_xlabel("Mean Inference Latency (ms / image)", fontsize=11, fontweight='bold')
ax.set_title("Computational Efficiency Benchmark (Inference Latency)", fontsize=12, fontweight='bold', pad=12)
ax.grid(axis='x', linestyle='--', alpha=0.6)

for bar in bars:
    w = bar.get_width()
    ax.annotate(f"{w:.3f} ms", xy=(w, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords="offset points", ha='left', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, max(avg_latencies) * 1.35)
textstr = f"Model Offline Training Times:\n• PCA ({pca_mname}): {pca_fit_time:.2f} s\n• CDAE (10 Epochs): {cdae_total_train_time:.1f} s ({cdae_total_train_time/60:.1f} min)"
props = dict(boxstyle='round,pad=0.5', facecolor='#f7f9fa', edgecolor='#cccccc', alpha=0.95)
ax.text(0.52, 0.22, textstr, transform=ax.transAxes, fontsize=9.5, verticalalignment='top', bbox=props)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'runtime_comparison.png'), bbox_inches='tight')
plt.close(fig)

# 9. Residual Maps
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5), dpi=300)
res_idx = 1
res_cond = "Gaussian (sigma=25)"
clean_ref = X_test_clean[res_idx]

res_items = [
    ("Noisy Input", denoised_cache[res_cond]["Noisy Input"][res_idx]),
    ("Gaussian Filter", denoised_cache[res_cond]["Gaussian Filter"][res_idx]),
    ("Median Filter", denoised_cache[res_cond]["Median Filter"][res_idx]),
    ("Bilateral Filter", denoised_cache[res_cond]["Bilateral Filter"][res_idx]),
    (pca_mname, denoised_cache[res_cond][pca_mname][res_idx]),
    ("CDAE", denoised_cache[res_cond]["CDAE"][res_idx]),
]

for col, (title, img) in enumerate(res_items):
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=10, fontweight='bold')
    axes[0, col].axis('off')
    residual = np.abs(clean_ref - img)
    im_res = axes[1, col].imshow(residual, cmap='inferno', vmin=0, vmax=0.5)
    axes[1, col].set_title("Residual Error", fontsize=9, color='#d62728', fontweight='bold')
    axes[1, col].axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.3])
cb = fig.colorbar(im_res, cax=cbar_ax)
cb.set_label("Absolute Error |I_clean - I_est|", fontsize=8, fontweight='bold')
plt.suptitle("Restored Images (Top) and Absolute Residual Error Maps (Bottom)", fontsize=12, fontweight='bold', y=0.98)
fig.savefig(os.path.join(FIGURES_DIR, 'residual_difference_maps.png'), bbox_inches='tight')
plt.close(fig)

print("All 9 publication figures generated and saved to ./figures/")


All 9 publication figures generated and saved to ./figures/


In [10]:
# Discussion and Analysis (Programmatically Generated from Data)
print("=== EMPIRICAL DISCUSSION & FINDINGS ===")

# 1. PCA Analysis
print(f"\n1. PCA Subspace Dimensionality Analysis:")
print(f"   - Cumulative explained variance reaches 90% at k={k_90} and 95% at k={k_95}.")
print(f"   - The validation sweep across k in {k_candidates} shows that reconstruction PSNR peaks at k={best_k} (Val PSNR = {max(k_psnr_sweep):.2f} dB).")
print(f"   - For k > {best_k}, validation PSNR decreases (dropping to {k_psnr_sweep[-1]:.2f} dB at k=600). This indicates that components beyond k={best_k} increasingly capture noise variance rather than useful signal structure.")

# 2. Performance Comparison Across Noise Regimes
print(f"\n2. Comparative Performance Across Regimes:")
for cond in NOISE_CONDITIONS:
    cname = cond["name"]
    cond_df = df_benchmark[df_benchmark["Noise Condition"] == cname]
    noisy_row = cond_df[cond_df["Method"] == "Noisy Input"].iloc[0]
    best_p_name = best_by_psnr[cname]["method"]
    best_p_val = best_by_psnr[cname]["psnr"]
    best_s_name = best_by_ssim[cname]["method"]
    best_s_val = best_by_ssim[cname]["ssim"]
    print(f"   - {cname}: Baseline = {noisy_row['PSNR (dB)']:.2f} dB | Best PSNR = {best_p_name} ({best_p_val:.2f} dB) | Best SSIM = {best_s_name} ({best_s_val:.4f})")

# 3. Honest Assessment of Cases Worse Than Noisy Baseline
print(f"\n3. Conditions Where Denoising Methods Underperform the Noisy Baseline:")
underperformed = []
for cond in NOISE_CONDITIONS:
    cname = cond["name"]
    cond_df = df_benchmark[df_benchmark["Noise Condition"] == cname]
    noisy_psnr = cond_df[cond_df["Method"] == "Noisy Input"]["PSNR (dB)"].values[0]
    for _, row in cond_df[cond_df["Method"] != "Noisy Input"].iterrows():
        if row["PSNR (dB)"] < noisy_psnr:
            diff = noisy_psnr - row["PSNR (dB)"]
            underperformed.append(f"   - {cname}, {row['Method']}: PSNR {row['PSNR (dB)']:.2f} dB vs Noisy {noisy_psnr:.2f} dB (-{diff:.2f} dB)")
if underperformed:
    for u in underperformed:
        print(u)
    print("   Interpretation (Hypothesis): At mild noise (e.g., Gaussian sigma=15), fine edge detail and texture dominate over noise variance.")
    print("   Methods trained on mixed noise or using spatial averaging tend to over-smooth genuine structural edges, incurring larger squared error than the untouched noise.")
else:
    print("   All methods improved upon the noisy baseline.")

# 4. Latency Overview
print(f"\n4. Computational Latency Profile:")
for m in runtime_methods:
    sub = df_benchmark[df_benchmark["Method"] == m]["Runtime (ms/img)"]
    print(f"   - {m:18s}: Mean = {sub.mean():.3f} ms/img (Min: {sub.min():.3f} ms, Max: {sub.max():.3f} ms)")


=== EMPIRICAL DISCUSSION & FINDINGS ===

1. PCA Subspace Dimensionality Analysis:
   - Cumulative explained variance reaches 90% at k=82 and 95% at k=181.
   - The validation sweep across k in [10, 20, 35, 50, 75, 82, 120, 181, 220, 300, 400, 500, 600] shows that reconstruction PSNR peaks at k=400 (Val PSNR = 22.81 dB).
   - For k > 400, validation PSNR decreases (dropping to 22.38 dB at k=600). This indicates that components beyond k=400 increasingly capture noise variance rather than useful signal structure.

2. Comparative Performance Across Regimes:
   - Gaussian (sigma=15): Baseline = 26.09 dB | Best PSNR = Bilateral Filter (28.35 dB) | Best SSIM = CDAE (0.8566)
   - Gaussian (sigma=25): Baseline = 21.78 dB | Best PSNR = Bilateral Filter (24.52 dB) | Best SSIM = CDAE (0.8447)
   - Gaussian (sigma=50): Baseline = 16.10 dB | Best PSNR = CDAE (21.95 dB) | Best SSIM = CDAE (0.7731)
   - S&P (d=0.05): Baseline = 17.08 dB | Best PSNR = CDAE (23.37 dB) | Best SSIM = CDAE (0.8360)
   - S&